# Volume 12, Chapter 9: Adèles, idèles and Tate's thesis (Python companion)

Exercises first, solutions at the end. We compute with adèles of $\mathbb Q$ (fractional parts, the decomposition $\mathbb A=\mathbb Q+[0,1[\times\hat{\mathbb Z}$, the character $\psi$), integrate on $\mathbb Q_p$ by finite sums, check adelic Poisson summation and Tate's formula numerically, compute root numbers of Dirichlet characters and compare with PARI, and end with the computer-science face of "$\mathbb Q$ is discrete in $\mathbb A$": rational reconstruction, the trick that lets computer algebra systems solve rational problems with modular arithmetic. Requires `cypari2`.

In [ ]:
import math, cmath, random, time
from fractions import Fraction
import cypari2
pari = cypari2.Pari()
random.seed(9)

## Exercises

**P9.1 (★) Fractional parts and partial fractions.** Write `frac_p(q, p)` returning the $p$-adic fractional part $\{q\}_p\in\mathbb Z[\frac1p]\cap[0,1[$ of a rational $q$. Check Lemma 9.2 ($q-\sum_p\{q\}_p\in\mathbb Z$) and Proposition 9.9 ($\psi(q)=e^{-2\pi iq}\prod_pe^{2\pi i\{q\}_p}=1$) on $1000$ random rationals. Then write the decomposition of Theorem 9.3(b) for the adèle "rational $q$ at the finite places, real $x$ at infinity".

**P9.2 (★) Haar integrals on $\mathbb Z_p$ by finite sums.** A function on $\mathbb Z_p$ that only depends on $x\bmod p^k$ integrates as the average of its values over $\mathbb Z/p^k$ (each ball $a+p^k\mathbb Z_p$ has mass $p^{-k}$). Approximate $\int_{\mathbb Z_p}|x|_p^s\,dx$ by replacing $|x|_p$ with $\max(|x|_p,p^{-k})$, compare with formula (9.1), and deduce the local zeta integral $Z_p(\mathbf 1_{\mathbb Z_p},s)=(1-p^{-s})^{-1}$ (Proposition 9.6) for a few $p$ and $s$. Also check $\int_{\mathbb Z_p^\times}d^\times x=1$.

**P9.3 (★★) Adelic Poisson summation and Tate's formula, numerically.** (a) For $f=f_\infty\otimes\mathbf 1_{b+N\hat{\mathbb Z}}$ with $b=\frac13$, $N=\frac52$ and $f_\infty(x)=xe^{-\pi x^2/2}+e^{-\pi x^2}$, compute both sides of Theorem 9.12. (b) Using formula (9.3) for $f=e^{-\pi x^2}\otimes\mathbf 1_{\hat{\mathbb Z}}$, compute $\Lambda(s)$ at $s=\frac12+14i$ by numerical integration, and compare with $\pi^{-s/2}\Gamma(\frac s2)\zeta(s)$ from PARI.

**P9.4 (★★) Root numbers.** For every primitive Dirichlet character $\chi$ modulo $N\le40$, compute $\varepsilon(\chi)=\tau(\chi)/(i^a\sqrt N)$ (Corollary 9.21) and compare with PARI's `lfunrootres`. Check that $|\varepsilon|=1$, that $\varepsilon=1$ for real characters (Exercise 9.4), and that $\varepsilon(\overline\chi)=\overline{\varepsilon(\chi)}$ (why?). Then verify the functional equation $\Lambda_N(s,\chi)=\varepsilon(\chi)\Lambda_N(1-s,\overline\chi)$ at $s=0.3+2i$ for the character of order $4$ modulo $5$. How are the arguments of $\varepsilon(\chi)$ distributed on the circle for $N$ prime around $1000$?

**P9.5 (★★★) CS application: rational reconstruction.** By Theorem 9.3, a rational number is pinned down by being small at $\infty$ and integral at all primes but a few. Concretely: if $x=\frac ab$ with $|a|,|b|\le H$ and $\gcd(b,M)=1$, then $x$ is determined by $r=ab^{-1}\bmod M$ as soon as $M>2H^2$.
1. Prove the uniqueness claim, and implement *rational reconstruction*: run the extended Euclidean algorithm on $(M,r)$ and stop at the first remainder below $\sqrt{M/2}$ (Wang's algorithm). Explain why this is a shortest-vector search in the lattice $\{(a,b):a\equiv rb\bmod M\}$, a two-dimensional cousin of Minkowski's theorem.
2. Solve a random $n\times n$ integer linear system $Ax=y$ exactly in two ways: (i) *multimodular*: solve it modulo several word-size primes, combine by the Chinese remainder theorem, and reconstruct; (ii) *$p$-adic lifting* (Dixon): invert $A$ once modulo one prime $p$, then compute the $p$-adic expansion of $x$ digit by digit, $x_i=A^{-1}r_i\bmod p$, $r_{i+1}=(r_i-Ax_i)/p$, and reconstruct from $x\bmod p^k$. How much precision is needed (Hadamard's bound)? Compare the running times with Gaussian elimination over `Fraction` for $n=30,60,100$. In adelic terms, (i) approximates $x$ at many places, (ii) at one place to high precision.

## Solutions

### P9.1
**How to find it.** Write $q=\frac uv$ with $v=p^kw$, $p\nmid w$. Then $\{q\}_p=\frac c{p^k}$ where $c\equiv uw^{-1}\pmod{p^k}$, $0\le c<p^k$: indeed $q-\frac c{p^k}=\frac{u-cw}{p^kw}$, and $p^k\mid u-cw$.

In [ ]:
def frac_p(q, p):
    q = Fraction(q); v = q.denominator; k = 0
    while v % p == 0:
        v //= p; k += 1
    if k == 0:
        return Fraction(0)
    c = q.numerator * pow(v, -1, p**k) % p**k
    return Fraction(c, p**k)

def primes_of(n):
    return [int(p) for p in pari.factor(n)[0]] if n > 1 else []

def psi(q):
    q = Fraction(q)
    s = sum(frac_p(q, p) for p in primes_of(q.denominator))
    return cmath.exp(-2j * math.pi * q) * cmath.exp(2j * math.pi * s), s

worst = 0
for _ in range(1000):
    q = Fraction(random.randint(-10**6, 10**6), random.randint(1, 10**6))
    val, s = psi(q)
    assert (q - s).denominator == 1          # Lemma 9.2
    worst = max(worst, abs(val - 1))
print("max |psi(q) - 1| over 1000 rationals:", worst)
q = Fraction(5, 12)
print("5/12 =", " + ".join(str(frac_p(q, p)) for p in (2, 3)), "+", q - frac_p(q, 2) - frac_p(q, 3))

For the adèle $x=(x_\infty, q)$: $r=\sum_p\{q\}_p$, $n=\lfloor x_\infty-r\rfloor$, and $x=(r+n)+d$ with $d=(x_\infty-r-n,\ q-r-n)\in[0,1[\times\hat{\mathbb Z}$. The finite component $q-r-n$ is an integer, as it must be for a rational lying in $\hat{\mathbb Z}$.

In [ ]:
def decompose(x_inf, q):
    r = sum(frac_p(q, p) for p in primes_of(Fraction(q).denominator))
    n = math.floor(x_inf - r)
    return r + n, (x_inf - float(r + n), Fraction(q) - r - n)

print(decompose(math.pi, Fraction(5, 12)))
print(decompose(-2.5, Fraction(-7, 30)))

### P9.2
**How to find it.** Sum over the residues $a\bmod p^k$; the ball $a+p^k\mathbb Z_p$ has mass $p^{-k}$, and on it $|x|_p=|a|_p$ unless $a=0$, where we use $p^{-k}$ (the ball $p^k\mathbb Z_p$ contributes $O(p^{-k(1+\operatorname{Re}s)})$ anyway). The multiplicative measure is $(1-\frac1p)^{-1}\frac{dx}{|x|_p}$.

In [ ]:
def abs_p(a, p, k):
    if a == 0:
        return float(p)**(-k)
    v = 0
    while a % p == 0:
        a //= p; v += 1
    return float(p)**(-v)

def int_abs_s(p, s, k):
    return sum(abs_p(a, p, k)**s for a in range(p**k)) / p**k

for p, s, k in [(2, 1.0, 14), (3, 0.5, 9), (5, 2+1j, 6), (7, 0.2, 5)]:
    approx = int_abs_s(p, s, k)
    exact = (1 - 1/p) / (1 - p**(-1 - s))
    zeta_loc = int_abs_s(p, s - 1, k) / (1 - 1/p)
    units = sum(1 for a in range(p**k) if a % p) / p**k / (1 - 1/p)
    print(f"p={p} s={s}: integral {approx:.6f} vs {exact:.6f};  Z_p = {zeta_loc:.5f} vs {1/(1-p**(-s)):.5f};  vol(Z_p^x) = {units}")

For small $\operatorname{Re}s$ the local zeta integral converges slowly in $k$: the truncation error is of order $p^{-ks}$, the contribution of the tiny ball $p^k\mathbb Z_p$ weighted by $|x|^{s-1}$. This is no accident: $(1-p^{-s})^{-1}$ has a pole at $s=0$, and the integral feels it coming.

### P9.3
**How to find it.** (a) The left side is $\sum_mf_\infty(b+Nm)$ (proof of Theorem 9.12), the right side $\frac1N\sum_ke^{2\pi ibk/N}\hat f_\infty(\frac kN)$, with $\widehat{e^{-\pi\lambda x^2}}=\lambda^{-1/2}e^{-\pi y^2/\lambda}$ and $\widehat{xe^{-\pi\lambda x^2}}=-i\lambda^{-3/2}ye^{-\pi y^2/\lambda}$ (from formula (9.2) by scaling). (b) Here $F_f(t)=\sum_{n\ne0}e^{-\pi n^2t^2}$ and (9.3) reads $\Lambda(s)=\int_1^\infty F_f(t)(t^s+t^{1-s})\frac{dt}t+\frac1{s-1}-\frac1s$.

In [ ]:
lam = 0.5
f_inf = lambda x: x * math.exp(-math.pi * lam * x * x) + math.exp(-math.pi * x * x)
fhat_inf = lambda y: -1j * lam**-1.5 * y * math.exp(-math.pi * y * y / lam) + math.exp(-math.pi * y * y)
b, N = Fraction(1, 3), Fraction(5, 2)
lhs = sum(f_inf(float(b + N * m)) for m in range(-60, 61))
rhs = sum(cmath.exp(2j * math.pi * float(b * k / N)) * fhat_inf(float(k / N)) for k in range(-200, 201)) / float(N)
print("sum over Q of f     :", lhs)
print("sum over Q of f-hat :", rhs)

In [ ]:
pari.set_real_precision(30)
s = pari("1/2 + 14*I")
F = "(t)->2*sumpos(n=1, exp(-Pi*n^2*t^2))"
tate = pari(f"intnum(t=1, [oo, 1], ({F})(t)*(t^({s}) + t^(1-({s})))/t) + 1/(({s})-1) - 1/({s})")
direct = pari(f"Pi^(-({s})/2)*gamma(({s})/2)*zeta({s})")
print("formula (9.3):", tate)
print("PARI        :", direct)

The two agree to all digits shown. Near $s=\frac12+14i$ we are close to the first zero of $\zeta$, $\frac12+14.1347\ldots i$, so $\Lambda$ is small there: Tate's formula is also a practical way to compute $\zeta$ in the critical strip, and it is essentially how `lfun` works.

### P9.4
**How to find it.** In PARI, `G = znstar(N,1)` and `znchar`/`znconreychar` describe characters; `chareval(G, chi, n)` returns $\chi(n)$ as a fraction of a turn; `zncharconductor` gives the conductor (primitive iff it equals $N$); `lfunrootres(lfuncreate([G, chi]))[3]` is PARI's root number. Characters are listed by their Conrey labels $m\in(\mathbb Z/N)^\times$. For the conjugate: substituting $c\mapsto-c$ in $\overline{\tau(\chi)}=\sum\overline\chi(c)e^{-2\pi ic/N}$ gives $\tau(\overline\chi)=\chi(-1)\overline{\tau(\chi)}$, and $\chi(-1)=(-1)^a$ cancels against $\overline{i^a}=(-1)^ai^a$.

In [ ]:
def char_values(N, m):
    G = pari.znstar(N, 1); chi = pari.znconreychar(G, m)
    vals = {}
    for c in range(1, N + 1):
        if math.gcd(c, N) == 1:
            vals[c % N] = cmath.exp(2j * math.pi * float(pari.chareval(G, chi, c)))
    return G, chi, vals

def eps(N, vals):
    tau = sum(v * cmath.exp(2j * math.pi * c / N) for c, v in vals.items())
    a = 0 if abs(vals[N - 1] - 1) < 1e-9 else 1
    return tau / (1j**a * math.sqrt(N)), a

count, worst, real_ok = 0, 0, True
for N in range(3, 41):
    for m in range(1, N):
        if math.gcd(m, N) != 1:
            continue
        G, chi, vals = char_values(N, m)
        if int(pari('zncharconductor')(G, chi)) != N:
            continue
        e, a = eps(N, vals)
        w = complex(pari.lfunrootres(pari.lfuncreate([G, chi]))[2])
        worst = max(worst, abs(e - w), abs(abs(e) - 1)); count += 1
        if all(abs(v.imag) < 1e-9 for v in vals.values()):
            real_ok &= abs(e - 1) < 1e-9
        # conjugate character: Conrey label m^{-1}
        _, _, vbar = char_values(N, pow(m, -1, N))
        ebar, _ = eps(N, vbar)
        worst = max(worst, abs(ebar - e.conjugate()))
print(count, "primitive characters; max discrepancy", worst, "; real characters have eps = 1:", real_ok)

In [ ]:
pari.set_real_precision(25)
N, m = 5, 2
G, chi, vals = char_values(N, m)
e, a = eps(N, vals)
L = pari.lfuncreate([G, chi]); Lbar = pari.lfuncreate([G, pari.znconreychar(G, pow(m, -1, N))])
s = pari("0.3 + 2*I")
LamN = lambda Lf, z, a: pari(N)**(z/2) * pari.Pi()**(-(z + a)/2) * pari.gamma((z + a)/2) * pari.lfun(Lf, z)
print("Lambda_N(s, chi)              :", LamN(L, s, a))
print("eps * Lambda_N(1 - s, chi-bar):", e * complex(LamN(Lbar, 1 - s, a)))

In [ ]:
import matplotlib.pyplot as plt
args = []
for N in [1009, 1013, 1019, 1021]:
    G = pari.znstar(N, 1); g = int(pari.znprimroot(N))
    for j in range(1, N - 1):
        tau = sum(cmath.exp(2j * math.pi * (j * k / (N - 1) + pow(g, k, N) / N)) for k in range(N - 1))
        a = j % 2   # chi(-1) = chi(g^((N-1)/2)) = (-1)^j
        args.append(cmath.phase(tau / (1j**a * math.sqrt(N))))
plt.figure(figsize=(6, 3)); plt.hist(args, bins=40, density=True)
plt.axhline(1 / (2 * math.pi), color="k", lw=0.8); plt.xlabel("arg eps(chi)"); plt.show()

For $N$ prime, $\chi(g^k)=e^{2\pi ijk/(N-1)}$ runs over all characters as $j$ varies, and $\chi(-1)=(-1)^j$. The histogram is flat: the arguments of root numbers (normalised Gauss sums) are equidistributed on the circle, a theorem of Deligne and Katz for Gauss sums of a fixed prime modulus as $N\to\infty$ (via the Riemann hypothesis for curves over finite fields, Chapter 13 for the elementary case). Real characters are the exception, with $\varepsilon=1$ by Gauss's sign theorem.

### P9.5
**How to find it.** *Uniqueness.* If $\frac ab$ and $\frac{a'}{b'}$ (sizes $\le H$) have the same residue, then $M\mid ab'-a'b$, and $|ab'-a'b|\le2H^2<M$, so $ab'=a'b$. Adelically: $\delta=\frac{ab'-a'b}M$ is a rational number that is integral at every prime and smaller than $1$ at infinity, so $\delta=0$; this is the argument of Theorem 9.3(a). *Algorithm.* The extended Euclidean algorithm on $(M,r)$ produces pairs $(r_i,t_i)$ with $r_i\equiv t_ir\pmod M$, $|r_i|$ decreasing and $|t_i|$ increasing, $|r_it_{i+1}|\le M$; these are the successive minima of the lattice $\{(a,b):a\equiv rb\}$ of covolume $M$ (Lagrange-Gauss reduction is Euclid's algorithm), so the solution, if it exists, is among them. *Linear systems.* By Cramer, $x=\frac{\det A_i}{\det A}$, and Hadamard's bound $|\det|\le\prod\|\text{columns}\|$ bounds numerators and denominators by $H$; we need $\prod p>2H^2$.

In [ ]:
def ratrec(r, M):
    r0, r1, t0, t1 = M, r % M, 0, 1
    bound = math.isqrt(M // 2)
    while r1 > bound:
        q = r0 // r1
        r0, r1, t0, t1 = r1, r0 - q * r1, t1, t0 - q * t1
    if t1 == 0 or abs(t1) > bound or math.gcd(r1, abs(t1)) != 1:
        return None
    return Fraction(r1, t1)

M = 10**12 + 39
x = Fraction(-31415, 27183)
r = x.numerator * pow(x.denominator, -1, M) % M
print("residue", r, "-> reconstructed", ratrec(r, M))

In [ ]:
def solve_mod(A, y, p):
    n = len(A)
    Mx = [[a % p for a in row] + [yy % p] for row, yy in zip(A, y)]
    for c in range(n):
        piv = next(i for i in range(c, n) if Mx[i][c])
        Mx[c], Mx[piv] = Mx[piv], Mx[c]
        inv = pow(Mx[c][c], -1, p)
        Mx[c] = [v * inv % p for v in Mx[c]]
        for i in range(n):
            if i != c and Mx[i][c]:
                f = Mx[i][c]
                Mx[i] = [(vi - f * vc) % p for vi, vc in zip(Mx[i], Mx[c])]
    return [row[n] for row in Mx]

def solve_multimodular(A, y):
    n = len(A)
    cols = [math.sqrt(sum(A[i][j]**2 for i in range(n))) for j in range(n)] + [math.sqrt(sum(v * v for v in y))]
    logH = sum(math.log2(c) for c in cols) - min(math.log2(c) for c in cols)   # Hadamard, Cramer numerators
    need = 2 * logH + 2
    primes, M, res = [], 1, None
    p = 2**61 - 1
    while math.log2(M) < need:
        p = int(pari.precprime(p - 1))
        xs = solve_mod(A, y, p)
        if res is None:
            res, M = xs, p
        else:
            res = [int(pari.lift(pari.chinese(pari.Mod(a, M), pari.Mod(b, p)))) for a, b in zip(res, xs)]
            M *= p
        primes.append(p)
    return [ratrec(v, M) for v in res], len(primes)

def solve_fraction(A, y):
    n = len(A)
    Mx = [[Fraction(a) for a in row] + [Fraction(yy)] for row, yy in zip(A, y)]
    for c in range(n):
        piv = next(i for i in range(c, n) if Mx[i][c])
        Mx[c], Mx[piv] = Mx[piv], Mx[c]
        for i in range(c + 1, n):
            f = Mx[i][c] / Mx[c][c]
            Mx[i] = [vi - f * vc for vi, vc in zip(Mx[i], Mx[c])]
    x = [Fraction(0)] * n
    for i in reversed(range(n)):
        x[i] = (Mx[i][n] - sum(Mx[i][j] * x[j] for j in range(i + 1, n))) / Mx[i][i]
    return x

def inverse_mod(A, p):
    n = len(A)
    Mx = [[a % p for a in row] + [int(i == j) for j in range(n)] for i, row in enumerate(A)]
    for c in range(n):
        piv = next(i for i in range(c, n) if Mx[i][c])
        Mx[c], Mx[piv] = Mx[piv], Mx[c]
        inv = pow(Mx[c][c], -1, p)
        Mx[c] = [v * inv % p for v in Mx[c]]
        for i in range(n):
            if i != c and Mx[i][c]:
                f = Mx[i][c]
                Mx[i] = [(vi - f * vc) % p for vi, vc in zip(Mx[i], Mx[c])]
    return [row[n:] for row in Mx]

def solve_dixon(A, y, p=2**61 - 1):
    n = len(A)
    cols = [math.sqrt(sum(A[i][j]**2 for i in range(n))) for j in range(n)] + [math.sqrt(sum(v * v for v in y))]
    need = 2 * (sum(math.log2(c) for c in cols) - min(math.log2(c) for c in cols)) + 2
    Cinv = inverse_mod(A, p)
    r, x, pk, steps = list(y), [0] * n, 1, 0
    while math.log2(pk) < need:
        xi = [sum(c * v for c, v in zip(row, r)) % p for row in Cinv]
        Ax = [sum(a * v for a, v in zip(row, xi)) for row in A]
        r = [(ri - axi) // p for ri, axi in zip(r, Ax)]
        x = [X + pk * v for X, v in zip(x, xi)]
        pk *= p; steps += 1
    return [ratrec(v, pk) for v in x], steps

for n in [30, 60, 100]:
    A = [[random.randint(-10**6, 10**6) for _ in range(n)] for _ in range(n)]
    y = [random.randint(-10**6, 10**6) for _ in range(n)]
    t = time.time(); x1, k = solve_multimodular(A, y); t1 = time.time() - t
    t = time.time(); x3, steps = solve_dixon(A, y); t3 = time.time() - t
    t = time.time(); x2 = solve_fraction(A, y); t2 = time.time() - t
    print(f"n={n}: multimodular {t1:.2f}s ({k} primes), p-adic lifting {t3:.2f}s ({steps} digits),"
          f" Fraction elimination {t2:.2f}s; all agree: {x1 == x2 == x3}")

In pure Python the multimodular method is not faster than `Fraction` elimination (each of its many modular eliminations costs $n^3$ interpreted operations, while Python's big integers are implemented in C), but $p$-adic lifting is, and the gap grows with $n$: it does a single elimination, then only matrix-vector products with small numbers, while elimination over $\mathbb Q$ suffers from intermediate expression swell. In compiled libraries both modular methods beat rational elimination by large factors. This is how PARI, FLINT, Magma and Sage solve rational linear systems, compute determinants and gcds of polynomials, and it is a direct computational reading of Theorem 9.3: a rational number is a point of a discrete subgroup of $\mathbb A$, so a crude approximation at a few places (here: the primes of $M$, plus the size bound at $\infty$) determines it exactly.